# Configuration Guide

This tutorial explains the YAML configuration structure that NAT uses. Understanding this helps you read, write, and debug configurations.

## What You'll Learn

1. Configuration file structure
2. Component sections (functions, llms, workflow, etc.)
3. Component references (`_type`, `name`, references)
4. Environment variables in configs
5. Common configuration patterns


## Configuration Structure

A NAT configuration file has these main sections:

```yaml
# Individual functions (tools)
functions:
  tool_name:
    _type: tool_type
    # tool-specific config

# Groups of related functions
function_groups:
  group_name:
    _type: group_type
    include: [fn1, fn2]  # Optional: filter functions

# LLM configurations
llms:
  llm_name:
    _type: nim  # or openai
    model: model_name

# Embedders for RAG
embedders:
  embedder_name:
    _type: nim_embedder
    model: embedding_model

# Retrievers for RAG
retrievers:
  retriever_name:
    _type: milvus
    embedder_name: embedder_name

# Memory configurations
memory:
  memory_name:
    _type: mem0

# The main workflow
workflow:
  _type: react_agent  # or tool_call_agent, rewoo_agent
  llm_name: llm_name
  tool_names: [tool1, tool2]

# Optional: Evaluation config
evaluation:
  dataset:
    file_path: data.json
  evaluators:
    - _type: ragas
      metric: AnswerAccuracy

# Optional: Optimizer config
optimizer:
  eval_metrics:
    accuracy:
      maximize: true
```


## The `_type` Field

Every component has a `_type` field that identifies what kind of component it is:

```yaml
llms:
  my_llm:
    _type: nim              # NVIDIA NIM LLM
    model: meta/llama-3.3-70b-instruct

  other_llm:
    _type: openai           # OpenAI LLM
    model: gpt-4o
```

### Common `_type` Values

| Section | `_type` | Description |
|---------|---------|-------------|
| `llms` | `nim` | NVIDIA NIM |
| `llms` | `openai` | OpenAI |
| `functions` | `current_datetime` | Time tool |
| `function_groups` | `mcp_client` | MCP client |
| `function_groups` | `a2a_client` | A2A client |
| `workflow` | `react_agent` | ReAct agent |
| `workflow` | `tool_call_agent` | Tool calling agent |
| `workflow` | `rewoo_agent` | ReWOO agent |


## Component References

Components reference each other by name:

```yaml
llms:
  nim_llm:                    # This is the name
    _type: nim
    model: meta/llama-3.3-70b-instruct

functions:
  current_time:               # Tool name
    _type: current_datetime

workflow:
  _type: react_agent
  llm_name: nim_llm           # References the LLM by name
  tool_names:
    - current_time            # References the function by name
```


## Environment Variables

Use `${VAR_NAME}` syntax for environment variables:

```yaml
llms:
  openai_llm:
    _type: openai
    model: gpt-4o
    api_key: ${OPENAI_API_KEY}  # Reads from environment

authentication:
  kaggle:
    _type: api_key
    raw_key: ${KAGGLE_BEARER_TOKEN}
    auth_scheme: Bearer
```

This keeps secrets out of config files!


## Function Group `include` Filter

Filter which functions from a group are available:

```yaml
function_groups:
  calculator:
    _type: calculator
    include:                  # Only include these functions
      - add
      - multiply
    # exclude: [divide]       # Or exclude specific ones
```


## SDK to YAML Mapping

When you use the SDK, components are automatically serialized:

| SDK Class | YAML `_type` | Section |
|-----------|--------------|---------|
| `NimLLM` | `nim` | `llms` |
| `OpenAILLM` | `openai` | `llms` |
| `CurrentTimeTool` | `current_datetime` | `functions` |
| `MCPClient` | `mcp_client` | `function_groups` |
| `NatReActAgent` | `react_agent` | `workflow` |
| `ToolCallingAgent` | `tool_call_agent` | `workflow` |


In [1]:
import sys
from pathlib import Path

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))


In [2]:
# Example: See the generated YAML from SDK
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.tool.datetime_tools import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(model_name="meta/llama-3.3-70b-instruct", temperature=0.0, name="nim_llm")
tool = CurrentTimeTool(name="current_time")
agent = NatReActAgent(tools=[tool], llm=llm, verbose=True)
workflow = NatWorkflow(entrypoint=agent)

# Save and display
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)
config_path = config_dir / "config_guide_example.yaml"
workflow.save_to_config_file(config_path)

print("Generated YAML:\n")
with open(config_path) as f:
    print(f.read())


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Generated YAML:

functions:
  current_time:
    _type: current_datetime

llms:
  nim_llm:
    _type: nim
    model: meta/llama-3.3-70b-instruct
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_time



## CLI Override

Override config values from CLI:

```bash
# Override LLM temperature
nat run --config_file config.yaml \
    --override llms.nim_llm.temperature=0.7

# Override multiple values
nat run --config_file config.yaml \
    --override llms.nim_llm.temperature=0.7 \
    --override workflow.verbose=false
```

## Summary

✅ **Structure** - functions, llms, workflow sections  
✅ **`_type`** - Identifies component type  
✅ **References** - Components reference by name  
✅ **Environment variables** - `${VAR_NAME}` syntax  
✅ **CLI overrides** - `--override` flag  

## Next Steps

- **[09_evaluation.ipynb](./09_evaluation.ipynb)** - Evaluation configuration
- **[11_optimization.ipynb](./11_optimization.ipynb)** - Optimizer configuration
